# 商品匹配测试

这个 notebook 用来测试新版 RAG 商品匹配系统。

测试内容：

1. 检查配置和 Excel 文件。
2. 读取 `assets/spu.xlsx`。
3. 测试服务类型识别规则。
4. 调用 `match_product_tool` 做真实 Qwen embedding 匹配。
5. 查看本地 embedding 缓存。

注意：真实匹配会调用 Qwen embedding API。请先在 `.env` 配置：

```env
QWEN_EMBEDDING_API_KEY=你的 DashScope Key
```

如果没有配置 `QWEN_EMBEDDING_API_KEY`，代码会尝试使用 `OPENAI_API_KEY`。

In [ ]:
from pathlib import Path

from config.settings import settings

print("SPU_EXCEL_PATH:", settings.spu_excel_path)
print("EMBEDDING_CACHE_DIR:", settings.embedding_cache_dir)
print("QWEN_EMBEDDING_MODEL:", settings.qwen_embedding_model)
print("QWEN_EMBEDDING_BASE_URL:", settings.qwen_embedding_base_url)
print("has_qwen_key:", bool(settings.qwen_embedding_api_key))
print("has_openai_key:", bool(settings.openai_api_key))

excel_path = Path(settings.spu_excel_path)
print("excel_exists:", excel_path.exists(), excel_path.resolve())

## 1. 测试 Excel 加载

这一部分不会调用 embedding API，只检查 `assets/spu.xlsx` 是否能被正确读取，并展示前几条服务商品。

In [ ]:
from rag.spu_loader import SpuExcelLoader

records = SpuExcelLoader().load()
print("record_count:", len(records))

for record in records[:5]:
    print({
        "code": record.service_product_code,
        "name": record.service_product_name,
        "raw_service_type": record.raw_service_type,
        "service_order_type": record.service_order_type,
        "category": record.category,
        "related_category": record.related_category,
        "fault": record.fault_phenomenon,
    })

## 2. 测试服务类型识别

这一步也不会调用 embedding API。它检查用户话术能不能被规则识别成正确服务类型。

In [ ]:
from rag.product_matcher import ProductMatcher

retriever = ProductMatcher()

cases = [
    "马桶堵了",
    "卫生间水龙头漏水",
    "帮我装五金挂件",
    "量一下窗帘尺寸",
    "托管维修",
]

for text in cases:
    print(text, "=>", retriever.infer_service_type_hint(text))

## 3. 真实匹配测试

下面会调用 Qwen embedding API。

第一次运行会比较慢，因为系统会为 Excel 里的服务商品生成两组向量缓存：

- 服务商品名称
- 关联故障现象

如果结果为空，可以先把 `threshold` 从 `0.55` 降到 `0.45` 或 `0.4`。

In [ ]:
from tools.product_match import match_product_tool

recall_cases = [
    {
        "query": "888房马桶堵了",
        "product": "马桶",
        "fault": "堵塞",
        "area": "卫生间",
    },
    {
        "query": "帮我装五金挂件",
        "product": "五金挂件",
        "fault": "安装",
        "area": "客房",
    },
    {
        "query": "量一下窗帘尺寸",
        "product": "窗帘",
        "fault": "测量",
        "area": "客房",
    },
]

for case in recall_cases:
    result = match_product_tool.invoke({
        **case,
        "top_k": 3,
        "threshold": 0.45,
    })
    data = result.get("data", {})
    print("\nQUERY:", case["query"])
    print("status:", result.get("status"))
    print("error_code:", result.get("error_code"))
    print("message:", result.get("message"))
    print("service_type_hint:", data.get("service_type_hint"))
    print("best_match:")
    print(data.get("best_match"))
    print("candidate_count:", data.get("count"))

## 4. 查看 embedding 缓存

真实匹配运行成功后，`data/embedding_cache` 下应该会出现 `product_*.npy` 和对应的 `.json` 元数据文件。

In [ ]:
cache_dir = Path(settings.embedding_cache_dir)
print("cache_dir:", cache_dir.resolve())

if cache_dir.exists():
    for path in sorted(cache_dir.glob("product_*")):
        print(path.name, path.stat().st_size, "bytes")
else:
    print("缓存目录还不存在。请先运行真实匹配测试。")

In [ ]:
from rag.spu_loader import SpuExcelLoader
from rag.product_matcher import ProductMatcher

records = SpuExcelLoader("assets/spu.xlsx").load()
print("商品数量:", len(records))
print(records[0])

retriever = ProductMatcher(excel_path="assets/spu.xlsx")

for text in ["马桶堵了", "帮我装五金挂件", "量一下窗帘尺寸", "托管维修"]:
    print(text, "=>", retriever.infer_service_type_hint(text))

In [ ]:
from tools.product_match import match_product_tool
cases = [
    {
        "query": "888房马桶堵了",
        "product": "马桶",
        "fault": "堵塞",
        "area": "卫生间",
    },
    {
        "query": "帮我装五金挂件",
        "product": "五金挂件",
        "fault": "安装",
        "area": "客房",
    },
    {
        "query": "量一下窗帘尺寸",
        "product": "窗帘",
        "fault": "测量",
        "area": "客房",
    },
]
for case in cases:
    result = match_product_tool.invoke({
        **case,
        "top_k": 3,
        "threshold": 0.45,
    })
    print("\nQUERY:", case["query"])
    print(result["data"]["service_type_hint"])
    print(result["data"]["best_match"])